In [ ]:
#!pip install terratorch torch torchvision rasterio albumentations tqdm scikit-learn wandb


In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')
import gc

import torch
import numpy as np
import rasterio
from pathlib import Path
from sklearn.model_selection import train_test_split
import albumentations as A

# TerraTorch imports
from terratorch.cli_tools import LightningInferenceModel
from terratorch.datamodules.generic_pixel_wise_data_module import GenericPixelwiseRegressionDataModule
from terratorch.tasks import PixelwiseRegressionTask, SemanticSegmentationTask
from terratorch.datasets import GenericNonGeoPixelwiseRegressionDataset

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger, CSVLogger


In [ ]:
# ==========================================
# 1. CONFIGURAZIONE
# ==========================================

with open('config.json', 'r') as f:
    json_config = json.load(f)

CONFIG = {
    "DATA_DIR": json_config["paths"]['input_dir'],
    "MODEL_SAVE_DIR": Path(json_config["paths"]["model_save_path"]).parent,
    "LOGS_DIR": json_config["paths"]["logs_dir"],
    "NUM_CLASSES": json_config["data_specs"]["num_classes"],
    "IMG_SIZE": json_config["data_specs"]["img_size"],
    "BATCH_SIZE": json_config["training_params"]["batch_size"],
    "EPOCHS": json_config["training_params"]["num_epochs"],
    "NUM_WORKERS": json_config["training_params"]["num_workers"],
    "LEARNING_RATE": json_config["training_params"]["learning_rate"],
    "MEANS": json_config["data_specs"]["normalization"]["means"],
    "STDS": json_config["data_specs"]["normalization"]["stds"],
    # Prithvi-specifico
    "MODEL_NAME": "prithvi_eo_v2_600",
    "PRETRAINED_PATH": "ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL",
    "NUM_FRAMES": 1,  # Single timestamp (adattiamo temporale a singola immagine)
}

print("="*70)
print("🌍 CROP SEGMENTATION - Prithvi-100M (4GB GPU Optimized)")
print("="*70)
print(f"✓ Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"✓ Batch Size: {CONFIG['BATCH_SIZE']} x {CONFIG['GRAD_ACCUM_STEPS']} = {CONFIG['EFFECTIVE_BATCH_SIZE']}")
print(f"✓ Mixed Precision: {CONFIG['MIXED_PRECISION']}")
print("="*70 + "\n")
# Pulizia memoria iniziale
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None


Configurazione caricata per ibm-nasa-geospatial/Prithvi-EO-2.0-100M-TL
Device: cuda | Batch Reale: 16


In [ ]:
class CropTypeDataset(torch.utils.data.Dataset):
    """
    Dataset per Sentinel-2 10-band crop segmentation compatibile con TerraTorch.
    Prithvi si aspetta input (T, C, H, W) dove T=frames temporali
    """
    
    def __init__(self, file_list, root_dir, means, stds, augment=False, img_size=224):
        self.root_dir = Path(root_dir)
        self.img_dir = self.root_dir / "images"
        self.mask_dir = self.root_dir / "masks"
        self.files = file_list
        self.augment = augment
        self.img_size = img_size
        
        # Normalizzazione (10 bande)
        self.means = np.array(means, dtype=np.float32)
        self.stds = np.array(stds, dtype=np.float32)
        
        # Mappa 10-band a 6-band Prithvi (HLS bands)
        # Le tue bande: [B2, B3, B4, B8, B5, B6, B7, B8A, B11, B12]
        # Prithvi bands: [B2, B3, B4, B8A, B11, B12] (Blue, Green, Red, NIR, SWIR1, SWIR2)
        # Mapping: prendi indici [0,1,2,7,8,9] dal tuo stack
        self.band_indices = [0, 1, 2, 7, 8, 9]  # B2,B3,B4,B8A,B11,B12
        
        # Augmentation (solo geometrica - safe per multispettrale)
        self.transform = None
        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ShiftScaleRotate(
                    shift_limit=0.0625, 
                    scale_limit=0.1, 
                    rotate_limit=15, 
                    border_mode=0, 
                    p=0.5
                ),
            ])
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        filename = self.files[idx]
        
        # Carica immagine (10 bande)
        img_path = self.img_dir / filename
        with rasterio.open(img_path) as src:
            image = src.read().astype(np.float32)  # (10, H, W)
        
        # Carica maschera
        mask_path = self.mask_dir / filename
        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.int64)  # (H, W)
        
        # Seleziona solo le 6 bande compatibili con Prithvi
        image = image[self.band_indices, :, :]  # (6, H, W)
        
        # Transpose per albumentations
        image = np.transpose(image, (1, 2, 0))  # (H, W, 6)
        
        # Augmentation
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        # Normalizzazione (usa solo stats delle 6 bande selezionate)
        means_6band = self.means[self.band_indices].reshape(1, 1, 6)
        stds_6band = self.stds[self.band_indices].reshape(1, 1, 6)
        image = (image - means_6band) / (stds_6band + 1e-6)
        
        # Ritorna a (C, H, W)
        image = np.transpose(image, (2, 0, 1))  # (6, H, W)
        
        # Prithvi richiede (T, C, H, W) - aggiungi dimensione temporale
        image = np.expand_dims(image, axis=0)  # (1, 6, H, W)
        
        # Converti a tensor
        image = torch.from_numpy(image).float()
        mask = torch.from_numpy(mask).long()
        
        return {
            'image': image,  # (1, 6, H, W)
            'mask': mask     # (H, W)
        }

In [ ]:
# ==========================================
# 3. PYTORCH LIGHTNING DATA MODULE
# ==========================================

class CropTypeDataModule(pl.LightningDataModule):
    """DataModule per gestire train/val/test split"""
    
    def __init__(
        self,
        data_dir,
        means,
        stds,
        batch_size=4,
        num_workers=4,
        img_size=224,
        val_split=0.15,
        test_split=0.05
    ):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.means = means
        self.stds = stds
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.img_size = img_size
        self.val_split = val_split
        self.test_split = test_split
    
    def setup(self, stage=None):
        # Trova tutti i file
        img_dir = self.data_dir / "images"
        all_files = sorted([f.name for f in img_dir.glob("*.tif")])
        
        # Estrai labels per stratificazione
        try:
            labels = [int(f.split("_class")[1].split("_")[0]) for f in all_files]
            stratify = labels
        except:
            stratify = None
        
        # Split train/temp
        train_files, temp_files = train_test_split(
            all_files, 
            test_size=(self.val_split + self.test_split),
            random_state=42,
            stratify=stratify
        )
        
        # Split val/test
        if stratify:
            temp_labels = [int(f.split("_class")[1].split("_")[0]) for f in temp_files]
            val_files, test_files = train_test_split(
                temp_files,
                test_size=self.test_split/(self.val_split + self.test_split),
                random_state=42,
                stratify=temp_labels
            )
        else:
            val_files, test_files = train_test_split(
                temp_files,
                test_size=self.test_split/(self.val_split + self.test_split),
                random_state=42
            )
        
        print(f"\n📊 Dataset Split:")
        print(f"  Train: {len(train_files)} samples")
        print(f"  Val:   {len(val_files)} samples")
        print(f"  Test:  {len(test_files)} samples\n")
        
        # Crea datasets
        self.train_dataset = CropTypeDataset(
            train_files, self.data_dir, self.means, self.stds, 
            augment=True, img_size=self.img_size
        )
        self.val_dataset = CropTypeDataset(
            val_files, self.data_dir, self.means, self.stds, 
            augment=False, img_size=self.img_size
        )
        self.test_dataset = CropTypeDataset(
            test_files, self.data_dir, self.means, self.stds, 
            augment=False, img_size=self.img_size
        )
    
    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )
    
    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )
    
    def test_dataloader(self):
        return torch.utils.data.DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )


In [6]:
# Pesi calcolati approssimativamente sui tuoi dati
# Ordine: Class 0 (Background?), Class 1, Class 2, ... Class 8
# Se la Classe 0 non esiste o è background, gestiscila. Assumo qui classi 1-8.
# Normalizziamo affinché la classe più frequente (2) abbia peso ~1.0

# Counts: [?, 4670, 7057, 4556, 3596, 4751, 2436, 1678, 4273]
# Weights inversi (più è bassa la count, più alto il peso)
class_weights_list = [
    0.1,  # Class 0 (Se è background/nodata spesso si mette a 0 o basso)
    1.5,  # Class 1
    1.0,  # Class 2 (La più frequente = riferimento)
    1.55, # Class 3
    1.96, # Class 4
    1.48, # Class 5
    2.90, # Class 6
    4.20, # Class 7 (La più rara -> Peso più alto!)
    1.65  # Class 8
]

In [ ]:
# ==========================================
# 4. MODELLO PRITHVI + TASK CONFIG
# ==========================================

def create_prithvi_task(num_classes, learning_rate, class_weights=None):
    """
    Crea il task di semantic segmentation con Prithvi-EO-2.0-600M-TL
    """
    
    # Configurazione del modello
    model_args = {
        'decoder': 'UperNetDecoder',  # Decoder ottimizzato per ViT
        'decoder_args': {
            'in_channels': [1024, 1024, 1024, 1024],  # ViT-Huge features
            'num_classes': num_classes,
            'channels': 512,
            'pool_scales': [1, 2, 3, 6],
            'dropout_ratio': 0.1
        },
        'necks': [
            {
                'name': 'SelectIndices',
                'indices': [5, 11, 17, 23]  # Layer intermedi per skip connections
            },
            {
                'name': 'ReshapeTokensToImage'  # Riporta tokens a formato spaziale
            }
        ],
        'backbone': CONFIG["PRETRAINED_PATH"],
        'backbone_pretrained': True,
        'backbone_args': {
            'img_size': CONFIG["IMG_SIZE"],
            'num_frames': CONFIG["NUM_FRAMES"],
            'tubelet_size': 1,
            'patch_size': 16,
            'embed_dim': 1024,  # ViT-Huge
            'depth': 24,
            'num_heads': 16,
            'decoder_embed_dim': 512,
            'decoder_depth': 8,
            'decoder_num_heads': 16,
        }
    }
    
    # Loss function con pesi per class imbalance
    loss_args = {
        'loss': 'ce',  # CrossEntropy
        'ignore_index': -1
    }
    
    if class_weights is not None:
        loss_args['class_weights'] = class_weights
    
    # Crea il task
    task = SemanticSegmentationTask(
        model_args=model_args,
        model_factory="PrithviModelFactory",
        loss=loss_args['loss'],
        lr=learning_rate,
        optimizer="AdamW",
        optimizer_hparams={
            'weight_decay': 0.05,
            'betas': (0.9, 0.95)
        },
        scheduler="CosineAnnealingLR",
        scheduler_hparams={
            'T_max': CONFIG["EPOCHS"],
            'eta_min': 1e-6
        },
        freeze_backbone=False,  # Fine-tuning completo
        freeze_decoder=False,
        class_weights=class_weights,
        ignore_index=-1,
        num_classes=num_classes
    )
    
    return task


--- Inizio Procedura Training ---
Totale file trovati: 33018
Stratificazione attivata: le classi sono state estratte dai nomi dei file.
Training Set: 26414 immagini
Validation Set: 6604 immagini


TypeError: 'NoneType' object cannot be interpreted as an integer

In [ ]:
# ==========================================
# 5. TRAINING CON PYTORCH LIGHTNING
# ==========================================

def train():
    print("\n" + "="*70)
    print("🚀 AVVIO TRAINING")
    print("="*70 + "\n")
    
    # Crea directory
    CONFIG["MODEL_SAVE_DIR"].mkdir(parents=True, exist_ok=True)
    Path(CONFIG["LOGS_DIR"]).mkdir(parents=True, exist_ok=True)
    
    # DataModule
    datamodule = CropTypeDataModule(
        data_dir=CONFIG["DATA_DIR"],
        means=CONFIG["MEANS"],
        stds=CONFIG["STDS"],
        batch_size=CONFIG["BATCH_SIZE"],
        num_workers=CONFIG["NUM_WORKERS"],
        img_size=CONFIG["IMG_SIZE"],
        val_split=0.15,
        test_split=0.05
    )
    
    # Class weights per imbalance (dal tuo config originale)
    class_weights = torch.tensor([
        0.1,   # Class 0 (background/nodata)
        1.5,   # Class 1
        1.0,   # Class 2 (baseline)
        1.55,  # Class 3
        1.96,  # Class 4
        1.48,  # Class 5
        2.90,  # Class 6
        4.20,  # Class 7 (rara!)
        1.65   # Class 8
    ], dtype=torch.float32)
    
    # Task (modello + loss + optimizer)
    task = create_prithvi_task(
        num_classes=CONFIG["NUM_CLASSES"],
        learning_rate=CONFIG["LEARNING_RATE"],
        class_weights=class_weights
    )
    
    # Callbacks
    checkpoint_callback = ModelCheckpoint(
        dirpath=CONFIG["MODEL_SAVE_DIR"],
        filename='prithvi-crop-{epoch:02d}-{val_loss:.4f}',
        monitor='val_loss',
        mode='min',
        save_top_k=3,
        save_last=True,
        verbose=True
    )
    
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=15,
        mode='min',
        verbose=True
    )
    
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    
    # Logger
    csv_logger = CSVLogger(
        save_dir=CONFIG["LOGS_DIR"],
        name="prithvi_crop_segmentation"
    )
    
    # Trainer
    trainer = pl.Trainer(
        max_epochs=CONFIG["EPOCHS"],
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='16-mixed',  # Mixed precision per performance
        callbacks=[checkpoint_callback, early_stop_callback, lr_monitor],
        logger=csv_logger,
        log_every_n_steps=10,
        gradient_clip_val=1.0,  # Stabilità training
        accumulate_grad_batches=4,  # Simula batch_size più grande
        deterministic=False,  # Massima velocità
        enable_progress_bar=True,
        enable_model_summary=True
    )
    
    # Training
    print("\n🔥 Inizio Fine-tuning Prithvi-EO-2.0-600M-TL...\n")
    trainer.fit(task, datamodule=datamodule)
    
    # Test sul best model
    print("\n📊 Valutazione su Test Set...\n")
    trainer.test(task, datamodule=datamodule, ckpt_path='best')
    
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETATO!")
    print(f"📁 Modelli salvati in: {CONFIG['MODEL_SAVE_DIR']}")
    print(f"📈 Log salvati in: {CONFIG['LOGS_DIR']}")
    print("="*70 + "\n")


In [ ]:
# ==========================================
# 6. INFERENCE SU NUOVE IMMAGINI
# ==========================================

def predict_single_image(model_checkpoint, image_path, output_path):
    """
    Esegue inferenza su una singola immagine
    """
    # Carica modello
    task = SemanticSegmentationTask.load_from_checkpoint(model_checkpoint)
    task.eval()
    task.freeze()
    
    # Carica immagine
    with rasterio.open(image_path) as src:
        image = src.read().astype(np.float32)
        meta = src.meta
    
    # Preprocessing (usa solo 6 bande)
    band_indices = [0, 1, 2, 7, 8, 9]
    image = image[band_indices, :, :]
    
    # Normalizza
    means = np.array(CONFIG["MEANS"])[band_indices].reshape(6, 1, 1)
    stds = np.array(CONFIG["STDS"])[band_indices].reshape(6, 1, 1)
    image = (image - means) / (stds + 1e-6)
    
    # Converti a tensor (1, 1, 6, H, W) - batch, time, channels, height, width
    image_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).float()
    
    # Inferenza
    with torch.no_grad():
        logits = task(image_tensor)
        prediction = torch.argmax(logits, dim=1).squeeze().cpu().numpy()
    
    # Salva risultato
    meta.update(dtype=rasterio.uint8, count=1)
    with rasterio.open(output_path, 'w', **meta) as dst:
        dst.write(prediction.astype(np.uint8), 1)
    
    print(f"✓ Prediction salvata: {output_path}")
    return prediction




In [ ]:
# ==========================================
# ESECUZIONE
# ==========================================

if __name__ == "__main__":
    # Training
    train()
    
    # Esempio di inferenza (opzionale)
    # best_model = CONFIG["MODEL_SAVE_DIR"] / "prithvi-crop-best.ckpt"
    # if best_model.exists():
    #     test_image = CONFIG["DATA_DIR"] / "images" / "test_sample.tif"
    #     output_pred = CONFIG["DATA_DIR"] / "predictions" / "test_sample_pred.tif"
    #     predict_single_image(best_model, test_image, output_pred)